In [4]:
"""
Memory-Efficient Collaborative Filtering with Baseline Evaluation

This script integrates the SVD model pipeline with a simple
baseline "global average" model for performance comparison.

1.  Loads data using user-provided functions.
2.  Splits data into train/validation sets.
3.  Calculates a baseline MSE using the global average (user-provided).
4.  Trains a memory-efficient SVD model.
5.  Calculates the SVD model's MSE for comparison.
6.  Generates final predictions on the test set.
"""

import gzip
from collections import defaultdict
import os
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# --- Configuration ---
VALIDATION_SIZE = 0.2
N_COMPONENTS = 50
RANDOM_STATE = 42

# --- 1. User-Provided Helper Functions ---

def readGz(path):
  for l in gzip.open(path, 'rt'):
    yield eval(l)

def readCSV(path):
  f = gzip.open(path, 'rt')
  f.readline()  # Skip header
  for l in f:
    yield l.strip().split(',')

def read_test_csv(path):
    import os
    data = [d.split(',') for d in os.popen(f"cat {path}").read().split('\n')[1:-1]]
    return data

# --- User's Baseline Evaluation Function ---
def predict_global_avg(y_train, y_val):
    """
    Calculates the baseline MSE by predicting the mean
    of the training set for all validation items.
    
    Args:
        y_train (pd.Series): The 'rating' column from the train_set
        y_val (pd.Series): The 'rating' column from the val_set
    """
    mu = np.mean(y_train)
    y_pred = np.array([mu for d in y_val])
    print(f"Validation set shape: {y_pred.shape}")
    y_test = np.array(y_val)
    mse = np.mean((y_test - y_pred)**2)
    print(f"Baseline (Global Avg) MSE: {mse:.5f}")

# --- Helper Function for SVD Prediction ---
def predict_ratings(df, user_map, book_map, user_factors, item_factors, global_mean):
    """
    Generates predictions for a given DataFrame (val_set or test_df).
    This function is memory-safe and uses numpy for computation.
    """
    pred_df = df.copy()
    pred_df['user_idx'] = pred_df['userID'].map(user_map)
    pred_df['book_idx'] = pred_df['bookID'].map(book_map)
    
    # Initialize all predictions to the global mean (baseline)
    pred_df['predicted_rating'] = global_mean
    
    valid_rows = pred_df.dropna(subset=['user_idx', 'book_idx'])
    
    if not valid_rows.empty:
        valid_user_indices = valid_rows['user_idx'].astype(int).values
        valid_book_indices = valid_rows['book_idx'].astype(int).values

        # Vectorized Numpy Computation
        test_user_vecs = user_factors[valid_user_indices]
        test_book_vecs = item_factors[valid_book_indices]
        centered_preds = np.sum(test_user_vecs * test_book_vecs, axis=1)
        final_preds = centered_preds + global_mean
        
        pred_df.loc[valid_rows.index, 'predicted_rating'] = final_preds
        
    return pred_df

In [5]:
# --- 2. Data Loading (from user outline) ---
print("\n--- 2. Loading Data ---")
data_train_ti = [d for d in readCSV('train_Interactions.csv.gz')]
data_test_rating = read_test_csv('pairs_Rating.csv')
# --- 3. DataFrame Conversion (from user outline) ---
print("\n--- 3. Converting to Memory-Efficient DataFrames ---")

# Convert list-of-lists to DataFrame
train_df = pd.DataFrame(
    data_train_ti, columns=['userID', 'bookID', 'rating']
)
# Cast types for memory efficiency
train_df['userID'] = train_df['userID'].astype('category')
train_df['bookID'] = train_df['bookID'].astype('category')
train_df['rating'] = train_df['rating'].astype('float32')

test_df = pd.DataFrame(
    data_test_rating, columns=['userID', 'bookID']
)
test_df['userID'] = test_df['userID'].astype('category')
test_df['bookID'] = test_df['bookID'].astype('category')

print(f"Loaded {len(train_df)} total training interactions.")
print(f"Loaded {len(test_df)} test pairs for rating prediction.")

# --- 4. Index Mapping (from user outline) ---
print("\n--- 4. Creating Sparse Index Mappings ---")

# Build maps from the *entire* training dataset
user_map = {uid: i for i, uid in enumerate(train_df['userID'].cat.categories)}
book_map = {bid: i for i, bid in enumerate(train_df['bookID'].cat.categories)}

n_users = len(user_map)
n_books = len(book_map)

print(f"Mapped {n_users} unique training users.")
print(f"Mapped {n_books} unique training books.")

# Apply integer indices to the main training DataFrame
train_df['user_idx'] = train_df['userID'].cat.codes
train_df['book_idx'] = train_df['bookID'].cat.codes

# --- 5. Train/Validation Split (from user outline) ---
print(f"\n--- 5. Splitting Training Data ---")

train_set, val_set = train_test_split(
    train_df,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE
)
print(f"New Training Set size: {len(train_set)}")
print(f"New Validation Set size: {len(val_set)}")

# --- 6. Baseline Model Evaluation (User Request) ---
print("\n--- 6. Baseline Model Evaluation (Global Average) ---")
# We pass the 'rating' columns (pandas.Series)
predict_global_avg(train_set['rating'], val_set['rating'])

# --- 7. SVD Model Training ---
print(f"\n--- 7. Training TruncatedSVD with {N_COMPONENTS} components ---")

# Calculate the mean *only from the new training set*
global_mean = train_set['rating'].mean()
print(f"SVD Global mean (from train_set): {global_mean:.4f}")

# Center the ratings *of the new training set*
centered_ratings = train_set['rating'] - global_mean

# Build the sparse matrix
ratings_sparse_csr = csr_matrix(
    (centered_ratings, (train_set['user_idx'], train_set['book_idx'])),
    shape=(n_users, n_books)
)
print(f"Created sparse matrix with shape {ratings_sparse_csr.shape}.")

svd_model = TruncatedSVD(
    n_components=N_COMPONENTS,
    random_state=RANDOM_STATE
)

# Fit the SVD model on the sparse training data
user_factors = svd_model.fit_transform(ratings_sparse_csr)
item_factors_T = svd_model.components_
item_factors = item_factors_T.T # Shape: (n_books, n_components)

print(f"Created user-factor matrix: {user_factors.shape}")
print(f"Created item-factor matrix: {item_factors.shape}")

# --- 8. SVD Model Evaluation (on Validation Set) ---
print("\n--- 8. SVD Model Evaluation ---")

# Generate predictions for the validation set
val_set_preds = predict_ratings(
    val_set, user_map, book_map, 
    user_factors, item_factors, global_mean
)

# Calculate and print the SVD model's MSE
val_mse_svd = mean_squared_error(
    val_set_preds['rating'], 
    val_set_preds['predicted_rating']
)
print(f"SVD Model MSE (Validation): {val_mse_svd:.5f}")

# --- 9. Final Test Set Predictions ---
print("\n--- 9. Generating Final Test Set Predictions ---")



--- 2. Loading Data ---

--- 3. Converting to Memory-Efficient DataFrames ---
Loaded 200000 total training interactions.
Loaded 10000 test pairs for rating prediction.

--- 4. Creating Sparse Index Mappings ---
Mapped 27945 unique training users.
Mapped 6688 unique training books.

--- 5. Splitting Training Data ---
New Training Set size: 160000
New Validation Set size: 40000

--- 6. Baseline Model Evaluation (Global Average) ---
Validation set shape: (40000,)
Baseline (Global Avg) MSE: 1.71716

--- 7. Training TruncatedSVD with 50 components ---
SVD Global mean (from train_set): 3.6857
Created sparse matrix with shape (27945, 6688).
Created user-factor matrix: (27945, 50)
Created item-factor matrix: (6688, 50)

--- 8. SVD Model Evaluation ---
SVD Model MSE (Validation): 1.71611

--- 9. Generating Final Test Set Predictions ---


In [7]:
"""
Hybrid Recommender System (SVD + Feature Engineering)

This script implements a 2-stage hybrid model, which is memory-efficient
and robust.

1.  MODEL 1 (SVD): A TruncatedSVD model is trained on the sparse
    user-item matrix. Its *only* purpose is to generate a
    powerful 'svd_prediction' feature.

2.  MODEL 2 (Linear Regression): A LinearRegression model is trained
    using a rich feature set, including:
    -   Statistical aggregates (mean, std, count, iqr, mode)
        for both users and books.
    -   The 'svd_prediction' from Model 1.

This hybrid approach almost always outperforms a single model alone.
"""

import gzip
from collections import defaultdict
import os
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

# --- Configuration ---
VALIDATION_SIZE = 0.2
N_COMPONENTS = 50  # Latent factors for SVD (Model 1)
RANDOM_STATE = 42

# --- 1. User-Provided Helper Functions ---

def readGz(path):
  for l in gzip.open(path, 'rt'):
    yield eval(l)

def readCSV(path):
  f = gzip.open(path, 'rt')
  f.readline()  # Skip header
  for l in f:
    yield l.strip().split(',')

def read_test_csv(path):
    import os
    data = [d.split(',') for d in os.popen(f"cat {path}").read().split('\n')[1:-1]]
    return data

# --- User's Baseline Evaluation Function ---
def predict_global_avg(y_train, y_val):
    """Calculates the baseline MSE by predicting the mean."""
    mu = np.mean(y_train)
    y_pred = np.full(len(y_val), mu)
    mse = mean_squared_error(y_val, y_pred)
    print(f"Baseline (Global Avg) MSE: {mse:.5f}")

# --- Helper Function for SVD Prediction ---
def get_svd_predictions(df, user_map, book_map, user_factors, item_factors, global_mean):
    """
    Generates *only* the SVD predictions.
    This is used as a feature for Model 2.
    """
    # Map string IDs to integer indices
    user_idx = df['userID'].map(user_map)
    book_idx = df['bookID'].map(book_map)
    
    # Initialize all predictions to the global mean
    preds = np.full(len(df), global_mean)
    
    # Identify valid rows (non-cold-start)
    valid_mask = ~user_idx.isna() & ~book_idx.isna()
    valid_user_indices = user_idx[valid_mask].astype(int).values
    valid_book_indices = book_idx[valid_mask].astype(int).values
    
    if len(valid_user_indices) > 0:
        # Vectorized Numpy Computation
        test_user_vecs = user_factors[valid_user_indices]
        test_book_vecs = item_factors[valid_book_indices]
        centered_preds = np.sum(test_user_vecs * test_book_vecs, axis=1)
        
        # Add mean back and assign to the correct rows
        preds[valid_mask] = centered_preds + global_mean
        
    return preds

# --- Helper Function for Hybrid Feature Creation ---
def create_hybrid_features(df, user_stats, book_stats, global_fill_stats, svd_components):
    """
    Builds the final feature matrix for Model 2 by merging
    all pre-computed statistics and the SVD prediction.
    """
    # Start with indices
    X = df[['user_idx', 'book_idx']].copy()
    
    # 1. Merge User and Book Statistical Features
    X = X.merge(user_stats, on='user_idx', how='left')
    X = X.merge(book_stats, on='book_idx', how='left')
    
    # 2. Add SVD Prediction as a feature
    X['svd_pred'] = get_svd_predictions(df=df, **svd_components)
    
    # 3. Handle Cold Starts (Fill NaNs)
    # Any user/book not in the train_set will have NaNs
    # We fill them with the global stats from the training set
    X.fillna(global_fill_stats, inplace=True)
    
    # 4. Drop ID columns to create the final matrix
    X.drop(columns=['user_idx', 'book_idx'], inplace=True)
    
    return X

In [8]:
# --- 2. Data Loading ---
print("\n--- 2. Loading Data ---")
data_train_ti = [d for d in readCSV('train_Interactions.csv.gz')]
data_test_rating = read_test_csv('pairs_Rating.csv')

# --- 3. DataFrame Conversion ---
print("\n--- 3. Converting to Memory-Efficient DataFrames ---")
train_df = pd.DataFrame(
    data_train_ti, columns=['userID', 'bookID', 'rating']
)
train_df['userID'] = train_df['userID'].astype('category')
train_df['bookID'] = train_df['bookID'].astype('category')
train_df['rating'] = train_df['rating'].astype('float32')

test_df = pd.DataFrame(
    data_test_rating, columns=['userID', 'bookID']
)
test_df['userID'] = test_df['userID'].astype('category')
test_df['bookID'] = test_df['bookID'].astype('category')

# --- 4. Index Mapping ---
print("\n--- 4. Creating Sparse Index Mappings ---")
user_map = {uid: i for i, uid in enumerate(train_df['userID'].cat.categories)}
book_map = {bid: i for i, bid in enumerate(train_df['bookID'].cat.categories)}
n_users = len(user_map)
n_books = len(book_map)

# Apply integer indices to *both* dataframes
train_df['user_idx'] = train_df['userID'].cat.codes
train_df['book_idx'] = train_df['bookID'].cat.codes
test_df['user_idx'] = test_df['userID'].map(user_map).astype(int) if 'userID' in test_df else -1
test_df['book_idx'] = test_df['bookID'].map(book_map).astype(int) if 'bookID' in test_df else -1


# --- 5. Train/Validation Split ---
print(f"\n--- 5. Splitting Training Data ---")
train_set, val_set = train_test_split(
    train_df,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE
)
print(f"New Training Set size: {len(train_set)}")
print(f"New Validation Set size: {len(val_set)}")

# --- 6. Baseline Model Evaluation (Global Average) ---
print("\n--- 6. Baseline Model Evaluation ---")
predict_global_avg(train_set['rating'], val_set['rating'])

# --- 7. Model 1 Training (SVD) ---
print(f"\n--- 7. Training Model 1 (SVD) as Feature Generator ---")

global_mean = train_set['rating'].mean()
centered_ratings = train_set['rating'] - global_mean

ratings_sparse_csr = csr_matrix(
    (centered_ratings, (train_set['user_idx'], train_set['book_idx'])),
    shape=(n_users, n_books)
)

svd_model = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
user_factors = svd_model.fit_transform(ratings_sparse_csr)
item_factors = svd_model.components_.T  # Shape: (n_books, n_components)

# Store SVD components for the feature function
svd_components = {
    'user_map': user_map,
    'book_map': book_map,
    'user_factors': user_factors,
    'item_factors': item_factors,
    'global_mean': global_mean
}
print("SVD Model 1 (Feature Generator) is trained.")

# --- 8. Model 2 Feature Engineering (Your New Request) ---
print("\n--- 8. Engineering Features for Model 2 (Hybrid) ---")

# Define the aggregations
aggs = {
    'count': 'count',
    'mean': 'mean',
    'std': 'std',
    'mode': lambda x: x.mode().iloc[0] if not x.empty else np.nan,
    'q25': lambda x: x.quantile(0.25),
    'q75': lambda x: x.quantile(0.75)
}

# Calculate stats *only from the training set*
user_stats = train_set.groupby('user_idx')['rating'].agg(aggs)
book_stats = train_set.groupby('book_idx')['rating'].agg(aggs)

# Calculate IQR
user_stats['iqr'] = user_stats['q75'] - user_stats['q25']
book_stats['iqr'] = book_stats['q75'] - book_stats['q25']

# Clean up columns
user_stats.drop(columns=['q25', 'q75'], inplace=True)
book_stats.drop(columns=['q25', 'q75'], inplace=True)

# Rename columns to be specific (e.g., 'user_mean', 'book_count')
user_stats.columns = [f"user_{col}" for col in user_stats.columns]
book_stats.columns = [f"book_{col}" for col in book_stats.columns]

# Fill NaNs *in the stats tables*. (e.g., std is NaN for 1 rating)
# A std/iqr of 0 is a sensible default for a single-rating item.
user_stats.fillna({'user_std': 0, 'user_iqr': 0}, inplace=True)
book_stats.fillna({'book_std': 0, 'book_iqr': 0}, inplace=True)

# --- Create Fill-Values for Cold Starts ---
# These are the *global* stats, used to fill in for users/books
# in the validation/test set that were *not* in the train_set.
global_fill_stats = {
    'user_count': 0,
    'user_mean': global_mean,
    'user_std': train_set['rating'].std(),
    'user_mode': train_set['rating'].mode().iloc[0],
    'user_iqr': train_set['rating'].quantile(0.75) - train_set['rating'].quantile(0.25),
    'book_count': 0,
    'book_mean': global_mean,
    'book_std': train_set['rating'].std(),
    'book_mode': train_set['rating'].mode().iloc[0],
    'book_iqr': train_set['rating'].quantile(0.75) - train_set['rating'].quantile(0.25)
}
print("Feature lookups (mean, std, iqr, etc.) created.")

# --- 9. Build Train/Validation Sets for Model 2 ---
print("\n--- 9. Building Final Train/Validation Matrices ---")

# Create training set for Model 2
X_train_linear = create_hybrid_features(
    train_set, user_stats, book_stats, global_fill_stats, svd_components
)
y_train_linear = train_set['rating']

# Create validation set for Model 2
X_val_linear = create_hybrid_features(
    val_set, user_stats, book_stats, global_fill_stats, svd_components
)
y_val_true = val_set['rating']

print(f"Hybrid feature matrix shape: {X_train_linear.shape}")

# --- 10. Train and Evaluate Model 2 (Linear Regression) ---
print("\n--- 10. Training and Evaluating Model 2 (Linear Regression) ---")

lin_reg = LinearRegression()
lin_reg.fit(X_train_linear, y_train_linear)

# --- Evaluate all models on Validation Set ---

# 1. SVD-Only Model MSE
# We get this from the 'svd_pred' column in the validation matrix
y_val_pred_svd_only = X_val_linear['svd_pred']
val_mse_svd_only = mean_squared_error(y_val_true, y_val_pred_svd_only)
print(f"Model 1 (SVD Only) MSE:   {val_mse_svd_only:.5f}")

# 2. Hybrid Linear Model MSE
y_val_pred_hybrid = lin_reg.predict(X_val_linear)
val_mse_hybrid = mean_squared_error(y_val_true, y_val_pred_hybrid)
print(f"Model 2 (Hybrid) MSE:     {val_mse_hybrid:.5f}")


--- 2. Loading Data ---

--- 3. Converting to Memory-Efficient DataFrames ---

--- 4. Creating Sparse Index Mappings ---


IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [12]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Hybrid Recommender System (SVD + Feature Engineering)

[FIXED] This script corrects the 'IntCastingNaNError' by removing
the .astype(int) conversion when mapping test set indices.
The pipeline is already designed to handle the resulting NaN values
for cold-start users.
"""

import gzip
from collections import defaultdict
import os
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

# --- Configuration ---
VALIDATION_SIZE = 0.2
N_COMPONENTS = 9  # Latent factors for SVD (Model 1)
RANDOM_STATE = 42

# --- 1. User-Provided Helper Functions ---

def readGz(path):
  for l in gzip.open(path, 'rt'):
    yield eval(l)

def readCSV(path):
  f = gzip.open(path, 'rt')
  f.readline()  # Skip header
  for l in f:
    yield l.strip().split(',')

def read_test_csv(path):
    import os
    data = [d.split(',') for d in os.popen(f"cat {path}").read().split('\n')[1:-1]]
    return data


# --- User's Baseline Evaluation Function ---
def predict_global_avg(y_train, y_val):
    """Calculates the baseline MSE by predicting the mean."""
    mu = np.mean(y_train)
    y_pred = np.full(len(y_val), mu)
    mse = mean_squared_error(y_val, y_pred)
    print(f"Baseline (Global Avg) MSE: {mse:.5f}")

# --- Helper Function for SVD Prediction ---
def get_svd_predictions(df, user_map, book_map, user_factors, item_factors, global_mean):
    """
    Generates *only* the SVD predictions.
    This is used as a feature for Model 2.
    """
    # Map string IDs to integer indices
    # This *will* create NaNs for cold-start users. This is correct.
    user_idx = df['userID'].map(user_map)
    book_idx = df['bookID'].map(book_map)
    
    # Initialize all predictions to the global mean
    preds = np.full(len(df), global_mean)
    
    # Identify valid rows (non-cold-start) by checking for NaNs
    valid_mask = ~user_idx.isna() & ~book_idx.isna()
    valid_user_indices = user_idx[valid_mask].astype(int).values
    valid_book_indices = book_idx[valid_mask].astype(int).values
    
    if len(valid_user_indices) > 0:
        # Vectorized Numpy Computation
        test_user_vecs = user_factors[valid_user_indices]
        test_book_vecs = item_factors[valid_book_indices]
        centered_preds = np.sum(test_user_vecs * test_book_vecs, axis=1)
        
        # Add mean back and assign to the correct rows
        preds[valid_mask] = centered_preds + global_mean
        
    return preds

# --- Helper Function for Hybrid Feature Creation ---
def create_hybrid_features(df, user_stats, book_stats, global_fill_stats, svd_components):
    """
    Builds the final feature matrix for Model 2 by merging
    all pre-computed statistics and the SVD prediction.
    """
    # Start with indices
    # df['user_idx'] and df['book_idx'] may contain NaNs. This is OK.
    X = df[['user_idx', 'book_idx']].copy()
    
    # 1. Merge User and Book Statistical Features
    # 'how=left' merge will correctly handle NaNs (they won't match)
    X = X.merge(user_stats, on='user_idx', how='left')
    X = X.merge(book_stats, on='book_idx', how='left')
    
    # 2. Add SVD Prediction as a feature
    X['svd_pred'] = get_svd_predictions(df=df, **svd_components)
    
    # 3. Handle Cold Starts (Fill NaNs)
    # Any user/book not in the train_set will have NaNs
    # We fill them with the global stats from the training set
    X.fillna(global_fill_stats, inplace=True)
    
    # 4. Drop ID columns to create the final matrix
    X.drop(columns=['user_idx', 'book_idx'], inplace=True)
    
    return X

# --- Main Execution ---    

print("\n--- 2. Loading Data ---")
data_train_ti = [d for d in readCSV('train_Interactions.csv.gz')]
data_test_rating = read_test_csv('pairs_Rating.csv')

# --- 3. DataFrame Conversion ---
print("\n--- 3. Converting to Memory-Efficient DataFrames ---")
train_df = pd.DataFrame(
    data_train_ti, columns=['userID', 'bookID', 'rating']
)
train_df['userID'] = train_df['userID'].astype('category')
train_df['bookID'] = train_df['bookID'].astype('category')
train_df['rating'] = train_df['rating'].astype('float32')

test_df = pd.DataFrame(
    data_test_rating, columns=['userID', 'bookID']
)
test_df['userID'] = test_df['userID'].astype('category')
test_df['bookID'] = test_df['bookID'].astype('category')

# --- 4. Index Mapping ---
print("\n--- 4. Creating Sparse Index Mappings ---")
user_map = {uid: i for i, uid in enumerate(train_df['userID'].cat.categories)}
book_map = {bid: i for i, bid in enumerate(train_df['bookID'].cat.categories)}
n_users = len(user_map)
n_books = len(book_map)

train_df['user_idx'] = train_df['userID'].cat.codes
train_df['book_idx'] = train_df['bookID'].cat.codes

# Map test_df indices (will create NaNs for cold starts, this is correct)
test_df['user_idx'] = test_df['userID'].map(user_map)
test_df['book_idx'] = test_df['bookID'].map(book_map)

# --- 5. Train/Validation Split ---
print(f"\n--- 5. Splitting Training Data ---")
train_set, val_set = train_test_split(
    train_df,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE
)
print(f"New Training Set size: {len(train_set)}")
print(f"New Validation Set size: {len(val_set)}")

# --- 6. Baseline Model Evaluation (Global Average) ---
print("\n--- 6. Baseline Model Evaluation ---")
predict_global_avg(train_set['rating'], val_set['rating'])

# --- 7. Model 1 Training (SVD) ---
print(f"\n--- 7. Training Model 1 (SVD) as Feature Generator ---")

global_mean = train_set['rating'].mean()
centered_ratings = train_set['rating'] - global_mean

ratings_sparse_csr = csr_matrix(
    (centered_ratings, (train_set['user_idx'], train_set['book_idx'])),
    shape=(n_users, n_books)
)

svd_model = TruncatedSVD(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
user_factors = svd_model.fit_transform(ratings_sparse_csr)
item_factors = svd_model.components_.T  # Shape: (n_books, n_components)

svd_components = {
    'user_map': user_map,
    'book_map': book_map,
    'user_factors': user_factors,
    'item_factors': item_factors,
    'global_mean': global_mean
}
print("SVD Model 1 (Feature Generator) is trained.")

# --- 8. Model 2 Feature Engineering (Your New Request) ---
print("\n--- 8. Engineering Features for Model 2 (Hybrid) ---")

# *** FIX: Using modern .agg() syntax to prevent SpecificationError ***

# Define aggregation functions
mode_func = lambda x: x.mode().iloc[0] if not x.empty and not x.mode().empty else np.nan
q25_func = lambda x: x.quantile(0.25)
q75_func = lambda x: x.quantile(0.75)

# Calculate stats *only from the training set*
# This syntax renames columns and avoids the error in one step.
user_stats = train_set.groupby('user_idx').agg(
    user_count=('rating', 'count'),
    user_mean=('rating', 'mean'),
    user_std=('rating', 'std'),
    user_mode=('rating', mode_func),
    user_q25=('rating', q25_func),
    user_q75=('rating', q75_func)
)

book_stats = train_set.groupby('book_idx').agg(
    book_count=('rating', 'count'),
    book_mean=('rating', 'mean'),
    book_std=('rating', 'std'),
    book_mode=('rating', mode_func),
    book_q25=('rating', q25_func),
    book_q75=('rating', q75_func)
)

# Calculate IQR directly
user_stats['user_iqr'] = user_stats['user_q75'] - user_stats['user_q25']
book_stats['book_iqr'] = book_stats['book_q75'] - book_stats['book_q25'] # <-- Bug fix

# Clean up intermediate quantile columns
user_stats.drop(columns=['user_q25', 'user_q75'], inplace=True)
book_stats.drop(columns=['book_q25', 'book_q75'], inplace=True)

# Fill NaNs *in the stats tables* (e.g., std is NaN for 1 rating)
user_stats.fillna({'user_std': 0, 'user_iqr': 0}, inplace=True)
book_stats.fillna({'book_std': 0, 'book_iqr': 0}, inplace=True)

# --- Create Fill-Values for Cold Starts ---
train_median = train_set['rating'].median() 
global_fill_stats = {
    'user_count': 0,
    'user_mean': global_mean,
    'user_std': train_set['rating'].std(),
    'user_mode': train_median,
    'user_iqr': train_set['rating'].quantile(0.75) - train_set['rating'].quantile(0.25),
    'book_count': 0,
    'book_mean': global_mean,
    'book_std': train_set['rating'].std(),
    'book_mode': train_median,
    'book_iqr': train_set['rating'].quantile(0.75) - train_set['rating'].quantile(0.25)
}

# Fill any remaining NaNs (e.g., from an empty mode)
user_stats.fillna(global_fill_stats, inplace=True)
book_stats.fillna(global_fill_stats, inplace=True)

print("Feature lookups (mean, std, iqr, etc.) created.")

# --- 9. Build Train/Validation Sets for Model 2 ---
print("\n--- 9. Building Final Train/Validation Matrices ---")

X_train_linear = create_hybrid_features(
    train_set, user_stats, book_stats, global_fill_stats, svd_components
)
y_train_linear = train_set['rating']

X_val_linear = create_hybrid_features(
    val_set, user_stats, book_stats, global_fill_stats, svd_components
)
y_val_true = val_set['rating']

print(f"Hybrid feature matrix shape: {X_train_linear.shape}")

# --- 10. Train and Evaluate Model 2 (Linear Regression) ---
print("\n--- 10. Training and Evaluating Model 2 (Linear Regression) ---")

lin_reg = LinearRegression()
lin_reg.fit(X_train_linear, y_train_linear)

# --- Evaluate all models on Validation Set ---

# 1. SVD-Only Model MSE
y_val_pred_svd_only = X_val_linear['svd_pred']
val_mse_svd_only = mean_squared_error(y_val_true, y_val_pred_svd_only)
print(f"Model 1 (SVD Only) MSE:   {val_mse_svd_only:.5f}")

# 2. Hybrid Linear Model MSE
y_val_pred_hybrid = lin_reg.predict(X_val_linear)
val_mse_hybrid = mean_squared_error(y_val_true, y_val_pred_hybrid)
print(f"Model 2 (Hybrid) MSE:     {val_mse_hybrid:.5f}")


--- 2. Loading Data ---

--- 3. Converting to Memory-Efficient DataFrames ---

--- 4. Creating Sparse Index Mappings ---

--- 5. Splitting Training Data ---
New Training Set size: 12
New Validation Set size: 4

--- 6. Baseline Model Evaluation ---
Baseline (Global Avg) MSE: 2.50000

--- 7. Training Model 1 (SVD) as Feature Generator ---
SVD Model 1 (Feature Generator) is trained.

--- 8. Engineering Features for Model 2 (Hybrid) ---
Feature lookups (mean, std, iqr, etc.) created.

--- 9. Building Final Train/Validation Matrices ---
Hybrid feature matrix shape: (12, 11)

--- 10. Training and Evaluating Model 2 (Linear Regression) ---
Model 1 (SVD Only) MSE:   2.50000
Model 2 (Hybrid) MSE:     2.50000
